# Finger-vein PAD - CNN feature extraction on Kaggle (GPU)

Runs **only the feature-extraction stage** of the project, with the project's own `fvpad` code and the exact manifests/folds
that were built in the main workspace (shipped in `outputs/manifests` and `outputs/folds`).

**Before running:** Settings → *Accelerator*: **GPU** (T4 or P100) · *Internet*: **On** (needed to download the three course
zips and the pretrained weights from GitHub). Upload `fvpad_kaggle.zip` with *Add Input → Upload → New Dataset* (any name).

**Output (Output tab → download):** `features_IDIAP.zip`, `features_SCUT.zip`, `features_PLUS.zip` (each < 400 MB) and
`extraction_log.txt`. Embeddings are computed for **all** eligible real images and **all** synthetic images (33,442 images × 5 CNNs).

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path
# locate the uploaded project (Kaggle extracts uploaded zips under /kaggle/input/<name>/...)
hits = glob.glob("/kaggle/input/**/fvpad/__init__.py", recursive=True)
assert hits, "Upload fvpad_kaggle.zip as a Kaggle dataset first (Add Input -> Upload)."
src = Path(hits[0]).parents[1]
PROJECT = Path("/kaggle/working/fvpad_project")
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(src, PROJECT)
print("project copied from", src)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9", "h5py"], check=True)

In [ ]:
# Download and extract the three course archives into the layout the code expects:
#   DATA/real  <- Data_FV_Spoofing_WS2025_26.zip   DATA/AE_Res <- AE_Res-diff.zip   DATA/VAE <- VAE-diff.zip
import urllib.request, zipfile
DATA = Path("/tmp/fvpad_data")          # outside /kaggle/working, so it is not saved as output
BASE = "https://www.cosy.sbg.ac.at/~uhl/"
for name, sub in [("Data_FV_Spoofing_WS2025_26.zip", "real"), ("AE_Res-diff.zip", "AE_Res"), ("VAE-diff.zip", "VAE")]:
    target = DATA / sub
    if target.exists() and any(target.iterdir()):
        print("already there:", target); continue
    zpath = Path("/tmp") / name
    if not zpath.exists():
        print("downloading", name, "..."); urllib.request.urlretrieve(BASE + name, zpath)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        assert not [n for n in zf.namelist() if n.startswith("/") or ".." in n.split("/")]
        zf.extractall(target)
    inner = target / "Data_FV_Spoofing_WS2025_26"
    if inner.exists():                  # flatten the top folder of the real-data archive
        for p in inner.iterdir(): shutil.move(str(p), str(target / p.name))
        inner.rmdir()
    zpath.unlink()
print(sorted(p.name for p in DATA.iterdir()))

In [ ]:
os.environ["FVPAD_DATA_ROOT"] = str(DATA)
sys.path.insert(0, str(PROJECT))
import torch
from fvpad import config, pipeline, features
print("device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (enable the GPU!)")
config.BATCH_SIZE = 64
tables = pipeline.prepare()             # loads the shipped manifests - does NOT rebuild them
todo = pipeline.images_to_embed(tables, all_synthetic=True)
missing = [p for p in todo.path if not (config.DATA_ROOT / p).exists()]
assert not missing, f"{len(missing)} manifest files not found, e.g. {missing[:3]}"
print(todo.groupby("dataset").size().rename("images"), "\nall", len(todo), "files found")

In [ ]:
import time
t0 = time.time()
pipeline.extract(tables, all_synthetic=True)        # same code as in the main project, just on the GPU
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
import numpy as np, platform, timm
log = [f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}",
       f"python {platform.python_version()}  torch {torch.__version__}  timm {timm.__version__}"]
for ds in config.DATASETS:
    zname = Path(f"/kaggle/working/features_{ds}.zip")
    with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as zf:
        for m in config.MODEL_NAMES:
            ids, X = features.load_cache(ds, m)
            assert np.isfinite(X).all()
            log.append(f"{ds:5s} {m:11s} {X.shape}  mean norm {np.linalg.norm(X, axis=1).mean():.3f}")
            zf.write(features.cache_path(ds, m), arcname=f"{ds}/{m}.npz")
    log.append(f"{zname.name}: {zname.stat().st_size / 1e6:.0f} MB")
Path("/kaggle/working/extraction_log.txt").write_text("\n".join(log))
print("\n".join(log))
shutil.rmtree(PROJECT)      # keep only the zips + log as notebook output